# SDH exp_020 — H0 Selective-EB 3-seed 독립 재현

보고된 CV `0.564797`을 파라미터 변경 없이 재현합니다. 이 노트북은 `test.csv`를 읽지 않습니다. 먼저 seeds `42/777/2024`를 재현하고, 성공한 경우에만 새로운 seeds `31415/52/62`를 실행합니다.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

def find_root(start):
    for path in (start, *start.parents):
        if (path / 'data' / 'raw' / 'train.csv').exists():
            return path
    raise FileNotFoundError('data/raw/train.csv가 있는 저장소 루트를 찾지 못했습니다.')

ROOT = find_root(Path.cwd().resolve())
EXP_DIR = ROOT / 'experiments' / 'SDH' / 'exp_020_selective_eb_reproduction'
RESULT_DIR = EXP_DIR / 'results'
RESULT_DIR.mkdir(exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

import provided_pipeline as provided
import oof_reproduction as exp

train = pd.read_csv(ROOT / 'data' / 'raw' / 'train.csv')
genes = [column for column in train.columns if column not in ('ID', 'SUBCLASS')]
assert list(train.columns) == ['ID', 'SUBCLASS', *genes]
assert int(train[genes].isna().sum().sum()) == 0
assert train['SUBCLASS'].nunique() == 26

print('root:', ROOT)
print('train:', train.shape, 'genes:', len(genes), 'classes:', train.SUBCLASS.nunique())
print('sklearn:', importlib.metadata.version('scikit-learn'))
print('lightgbm:', importlib.metadata.version('lightgbm'))
print('provided source sha256:', exp.source_sha256())

## 1. 동결 계약 확인

제공 원본과 exp20 평가기의 핵심 상수를 확인합니다. 이 셀은 모델을 학습하지 않습니다.

In [ ]:
assert exp.REPRODUCTION_SEEDS == (42, 777, 2024)
assert exp.FRESH_VALIDATION_SEEDS == (31415, 52, 62)
assert provided.SELECTIVE_MARGIN == 0.05
assert provided.SELECTIVE_LR_WEIGHT == 0.80
assert provided.H0_SPECIALIST_WEIGHT == 0.20
assert provided.REFERENCE_BLEND == 0.543679

provided_source = (EXP_DIR / 'provided_pipeline.py').read_text(encoding='utf-8')
evaluator_source = (EXP_DIR / 'oof_reproduction.py').read_text(encoding='utf-8')
assert 'concat([train' not in provided_source
assert 'test.csv' not in evaluator_source
for forbidden in ('V600E', 'R132H', 'KIPAN', 'GBMLGG'):
    assert forbidden not in provided_source
print('frozen parameter and leakage static audit: PASS')

## 2. 실행 helper

각 seed가 끝날 때 즉시 checkpoint를 저장합니다. 완료된 seed는 `results_by_seed`에 남으므로 같은 커널에서 다시 실행하지 마세요.

In [ ]:
results_by_seed = {}

def run_seed_once(seed, group):
    if seed in results_by_seed:
        print('already completed:', seed)
        return results_by_seed[seed]
    started = time.perf_counter()
    result = exp.run_seed(train, genes, seed=seed, verbose=True)
    results_by_seed[seed] = result
    exp.save_results([result], RESULT_DIR, run_name=f'{group}_seed{seed}')
    scores = pd.DataFrame(result.scores()).T.reset_index(names='variant')
    print(f'seed={seed} completed: {(time.perf_counter()-started)/60:.1f} min')
    display(scores)
    display(result.fold_metrics[['fold', 'h0_lr_specialist_f1', 'final_selective_eb_specialist_f1', 'selective_non_eb_rate', 'specialist_pairs']])
    return result

## 3. 재현 seed 42

가장 먼저 기존 H0와 `0.543679`의 차이를 검사합니다. `±0.001` 초과는 audit 경고로 남기고, 구현이 달라졌다고 볼 수준인 `±0.005` 초과에서만 중단합니다.

In [ ]:
run_seed_once(42, 'reproduction')

## 4. 재현 seeds 777 / 2024

두 셀을 하나씩 실행합니다. 한 seed가 끝날 때마다 결과가 저장됩니다.

In [ ]:
run_seed_once(777, 'reproduction')

In [ ]:
run_seed_once(2024, 'reproduction')

## 5. 보고값 `0.564797` 재현 판정

per-seed F1 평균과 OOF 확률평균 F1을 모두 계산합니다. 보고자가 어느 정의를 사용했는지도 동시에 확인할 수 있습니다.

In [ ]:
reproduction_results = [results_by_seed[seed] for seed in exp.REPRODUCTION_SEEDS]
reproduction_per_seed, reproduction_summary = exp.aggregate_results(reproduction_results)
reproduction_report = exp.save_results(reproduction_results, RESULT_DIR, run_name='reproduction_3seed')
display(reproduction_per_seed)
display(reproduction_summary)
print(json.dumps(reproduction_report, indent=2, ensure_ascii=False))
REPRODUCTION_MATCH = reproduction_report['matches_reported_as_mean'] or reproduction_report['matches_reported_as_probability_average']
print('reported 0.564797 reproduced:', REPRODUCTION_MATCH)

## 6. Fresh seeds `31415 / 52 / 62`

위 재현이 성공했을 때만 설정을 그대로 고정해 실행합니다. 세 셀을 하나씩 실행하세요.

In [ ]:
if not REPRODUCTION_MATCH:
    raise RuntimeError('0.564797 재현에 실패했습니다. fresh seed 전에 차이를 확인하세요.')
print('reproduction PASS — fresh validation을 시작합니다.')

In [ ]:
run_seed_once(31415, 'fresh')

In [ ]:
run_seed_once(52, 'fresh')

In [ ]:
run_seed_once(62, 'fresh')

## 7. Fresh 3-seed 최종 판정

In [ ]:
fresh_results = [results_by_seed[seed] for seed in exp.FRESH_VALIDATION_SEEDS]
fresh_per_seed, fresh_summary = exp.aggregate_results(fresh_results)
fresh_report = exp.save_results(fresh_results, RESULT_DIR, run_name='fresh_3seed')
comparison = fresh_per_seed.pivot(index='seed', columns='variant', values='f1_macro').reset_index()
comparison['final_delta_vs_h0'] = comparison['final_selective_eb_specialist'] - comparison['h0_lr_specialist']
positive_folds = sum((item.fold_metrics['final_selective_eb_specialist_f1'] > item.fold_metrics['h0_lr_specialist_f1']).sum() for item in fresh_results)
FRESH_PASS = bool((comparison['final_delta_vs_h0'] > 0).all() and positive_folds >= 8)
display(comparison)
display(fresh_summary)
print('positive folds:', positive_folds, '/15')
print('fresh validation PASS:', FRESH_PASS)
print(json.dumps(fresh_report, indent=2, ensure_ascii=False))

## 7. 검증된 6-seed OOF 결합

기존 제출 seeds `42/777/2024`와 fresh seeds `31415/52/62`의 OOF 확률을 동일 가중치로 평균합니다. 6-seed OOF가 두 3-seed 조합보다 모두 높고 누수·수렴 검사를 통과할 때만 다음 제출 셀을 실행합니다.

In [ ]:
SIX_SEEDS = (*exp.REPRODUCTION_SEEDS, *exp.FRESH_VALIDATION_SEEDS)
missing_seeds = [seed for seed in SIX_SEEDS if seed not in results_by_seed]
if missing_seeds:
    raise RuntimeError(f'먼저 위 seed 셀을 실행하세요: {missing_seeds}')

six_results = [results_by_seed[seed] for seed in SIX_SEEDS]
six_per_seed, six_summary = exp.aggregate_results(six_results)
final_variant = 'final_selective_eb_specialist'

def ensemble_f1(summary):
    return float(summary.loc[
        summary['variant'].eq(final_variant),
        'probability_averaged_oof_f1',
    ].iloc[0])

old3_score = ensemble_f1(reproduction_summary)
fresh3_score = ensemble_f1(fresh_summary)
six_score = ensemble_f1(six_summary)
all_audits_pass = all(item.audit['leakage_check'].all() for item in six_results)
warning_count = sum(item.convergence_warning_count for item in six_results)
SIX_SEED_PASS = (
    six_score > max(old3_score, fresh3_score)
    and all_audits_pass
    and warning_count == 0
)

six_comparison = pd.DataFrame({
    'ensemble': ['42/777/2024', '31415/52/62', 'all 6 seeds'],
    'seed_count': [3, 3, 6],
    'oof_macro_f1': [old3_score, fresh3_score, six_score],
    'delta_vs_submitted_3seed': [0.0, fresh3_score - old3_score, six_score - old3_score],
})
display(six_comparison)
display(six_summary)
print('all fold audits pass:', all_audits_pass)
print('convergence warnings:', warning_count)
print('6-seed submission gate:', 'PASS' if SIX_SEED_PASS else 'STOP')

six_summary.to_csv(RESULT_DIR / 'validated_6seed_summary.csv', index=False)
six_report = {
    'seeds': list(SIX_SEEDS),
    'submitted_3seed_oof_macro_f1': old3_score,
    'fresh_3seed_oof_macro_f1': fresh3_score,
    'six_seed_oof_macro_f1': six_score,
    'delta_vs_submitted_3seed': six_score - old3_score,
    'all_fold_audits_pass': all_audits_pass,
    'convergence_warning_count': warning_count,
    'submission_gate_pass': SIX_SEED_PASS,
    'test_read': False,
}
(RESULT_DIR / 'validated_6seed_report.json').write_text(
    json.dumps(six_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
if not SIX_SEED_PASS:
    raise RuntimeError('6-seed OOF가 채택 조건을 통과하지 못했습니다. 제출 셀을 실행하지 마세요.')

## 8. 6-seed full-train 제출 생성

바로 위 OOF gate가 PASS한 뒤에만 실행합니다. 각 seed는 full train에서 독립 학습되며 test는 변환·예측에만 사용됩니다. seed별 test 확률과 audit을 `results/`에 즉시 저장하므로 중간에 멈춰도 완료된 seed부터 재개합니다. 예상 시간은 기존 3-seed 제출 생성의 약 2배입니다.

In [ ]:
if not SIX_SEED_PASS:
    raise RuntimeError('6-seed OOF gate가 PASS하지 않았습니다.')

test = pd.read_csv(ROOT / 'data' / 'raw' / 'test.csv')
sample_submission = pd.read_csv(ROOT / 'data' / 'raw' / 'sample_submission.csv')
expected_classes = np.asarray(sorted(train['SUBCLASS'].unique()), dtype=object)
probability_columns = [f'prob__{label}' for label in expected_classes]
assert list(test.columns) == ['ID', *genes]
assert sample_submission['ID'].reset_index(drop=True).equals(test['ID'].reset_index(drop=True))

test_probabilities = []
seed_audits = []
for seed in SIX_SEEDS:
    probability_path = RESULT_DIR / f'exp20_6seed_seed{seed}_test_probability.csv'
    audit_path = RESULT_DIR / f'exp20_6seed_seed{seed}_test_probability.audit.json'
    if probability_path.exists() and audit_path.exists():
        cached = pd.read_csv(probability_path)
        audit = json.loads(audit_path.read_text(encoding='utf-8'))
        if list(cached.columns) != ['ID', *probability_columns]:
            raise RuntimeError(f'seed {seed} cache column contract mismatch')
        if not cached['ID'].reset_index(drop=True).equals(test['ID'].reset_index(drop=True)):
            raise RuntimeError(f'seed {seed} cache ID order mismatch')
        probability = cached[probability_columns].to_numpy(dtype=np.float64)
        if audit.get('model_seed') != seed or not audit.get('leakage_check', False):
            raise RuntimeError(f'seed {seed} cache audit mismatch')
        print(f'[cache] seed {seed} test probability loaded')
    else:
        print(f'[fit] seed {seed} full-train/test prediction')
        probability, classes, audit = provided.build_submission_probability(
            train, test, model_seed=seed
        )
        if not np.array_equal(classes, expected_classes):
            raise RuntimeError(f'seed {seed} class order mismatch')
        checkpoint = pd.DataFrame(probability, columns=probability_columns)
        checkpoint.insert(0, 'ID', test['ID'].to_numpy())
        checkpoint.to_csv(probability_path, index=False)
        audit_path.write_text(
            json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8'
        )
    if probability.shape != (len(test), len(expected_classes)):
        raise RuntimeError(f'seed {seed} probability shape mismatch')
    if not np.isfinite(probability).all() or not np.allclose(probability.sum(axis=1), 1.0, atol=1e-6):
        raise RuntimeError(f'seed {seed} probability normalization mismatch')
    test_probabilities.append(probability)
    seed_audits.append(audit)

six_probability = provided.average_seed_probabilities(test_probabilities)
old3_probability = provided.average_seed_probabilities(test_probabilities[:3])
submission = provided.make_submission_frame(
    sample_submission, test, six_probability, expected_classes
)
output_path = RESULT_DIR / 'submission_exp20_selective_eb_6seed_42_777_2024_31415_52_62.csv'
submission.to_csv(output_path, index=False)
changed_vs_old3 = int(np.sum(
    expected_classes[six_probability.argmax(axis=1)]
    != expected_classes[old3_probability.argmax(axis=1)]
))
submission_audit = {
    'seeds': list(SIX_SEEDS),
    'seed_weights': [1.0 / len(SIX_SEEDS)] * len(SIX_SEEDS),
    'weight_tuned': False,
    'six_seed_oof_macro_f1': six_score,
    'submitted_3seed_oof_macro_f1': old3_score,
    'changed_predictions_vs_submitted_3seed': changed_vs_old3,
    'test_role': 'transform_and_predict_only',
    'raw_train_test_concat': False,
    'leakage_check': bool(all(item['leakage_check'] for item in seed_audits)),
    'nan_as_mutation_count': int(max(item['nan_as_mutation_count'] for item in seed_audits)),
    'per_seed_audits': seed_audits,
    'output_file': str(output_path),
    'row_count': int(len(submission)),
}
submission_audit_path = output_path.with_suffix('.audit.json')
submission_audit_path.write_text(
    json.dumps(submission_audit, ensure_ascii=False, indent=2), encoding='utf-8'
)
if not pd.read_csv(output_path).equals(submission):
    raise RuntimeError('submission round-trip mismatch')
if not submission_audit['leakage_check'] or submission_audit['nan_as_mutation_count'] != 0:
    raise RuntimeError('submission safety audit failed')
print('submission:', output_path)
print('audit:', submission_audit_path)
print('changed predictions vs submitted 3-seed:', changed_vs_old3)
display(submission.head())